In [ ]:
!pip install pymupdf tokenizers konlpy python-mecab-ko matplotlib pandas

## 데이터 수집 - 국방일보 1년치 (2024-05 ~ 2025-04)

In [ ]:
from pathlib import Path
import re
import fitz  # PyMuPDF
fitz.TOOLS.mupdf_display_errors(False)

PDF_DIR  = Path("data/kookbang")
OUT_FILE = Path("data/kookbang_corpus2.txt")

def extract_pdf_text(pdf_path: Path) -> str:
    lines = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            words = page.get_text("words")  # (x0,y0,x1,y1, word, block, line, word_no)
            if not words:
                continue
            # y 좌표 기준으로 같은 줄 묶기
            line_dict = {}
            for w in words:
                y = round(w[1], 0)
                line_dict.setdefault(y, []).append((w[0], w[4]))  # (x, word)
            for y in sorted(line_dict):
                row = " ".join(word for _, word in sorted(line_dict[y]))
                row = row.strip()
                if row:
                    lines.append(row)
    return "\n".join(lines)

def main():
    pdf_files = sorted(PDF_DIR.glob("*.pdf"))
    if not pdf_files:
        print("PDF 파일이 없습니다.")
        return

    total_bytes = 0
    used_files  = 0

    with open(OUT_FILE, "w", encoding="utf-8") as out:
        for pdf_path in pdf_files:
            try:
                text = extract_pdf_text(pdf_path)
                if len(text) < 200:
                    print(f"[SKIP] {pdf_path.name} | 텍스트 너무 적음")
                    continue
                block       = f"### {pdf_path.name} ###\n{text}\n\n"
                block_bytes = len(block.encode("utf-8"))
                out.write(block)
                total_bytes += block_bytes
                used_files  += 1
                print(f"[OK] {pdf_path.name:30} | {block_bytes/1024/1024:.2f} MB | 누적 {total_bytes/1024/1024:.2f} MB")
            except Exception as e:
                print(f"[ERROR] {pdf_path.name} -> {e}")

    print(f"\n=== 완료 ===\n사용한 PDF 수: {used_files}\n최종 텍스트 용량: {total_bytes/1024/1024:.2f} MB")

main()

In [ ]:
import time, random
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

CORPUS_PATH = Path('data/kookbang_corpus2.txt')
SPLIT_DIR   = Path('data/splits')
MODEL_DIR   = Path('models')
RESULT_DIR  = Path('results')

for d in [SPLIT_DIR, MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# BPE: 학습 데이터 100% 기준 고유 음절 수 = 6,570개
#      merge가 발생하려면 vocab_size > 고유 문자 수 이어야 함
#      → 최솟값: 7,000
BPE_VOCAB_SIZE = 7000

# WordPiece: 연속 토큰에 ## 접두사를 붙이므로 동일 문자도 2종류로 저장
#            ('국' 과 '##국' 을 별개 토큰으로 취급)
#            → 실제 필요 vocab ≈ 고유 음절 수 × 2 ≈ 11,700개 이상
#            → 최솟값: 12,000
WP_VOCAB_SIZE  = 12000

RATIOS      = [0.10, 0.50, 1.00]
RATIO_NAMES = ['10%', '50%', '100%']

print(f'BPE vocab size: {BPE_VOCAB_SIZE}')
print(f'WP  vocab size: {WP_VOCAB_SIZE}')
print('설정 완료')

In [ ]:
with open(CORPUS_PATH, encoding='utf-8') as f:
    full_lines = [line.strip() for line in f if line.strip()]

random.seed(42)
random.shuffle(full_lines)

print(f'전체 줄 수: {len(full_lines):,}')
print(f'전체 용량: {CORPUS_PATH.stat().st_size / 1024 / 1024:.2f} MB')

# 평가셋 분리 (~15KB)
eval_lines, eval_chars = [], 0
for line in full_lines:
    eval_lines.append(line)
    eval_chars += len(line)
    if eval_chars >= 15000:
        break

train_lines = full_lines[len(eval_lines):]
eval_text   = '\n'.join(eval_lines)

with open('data/eval_set.txt', 'w', encoding='utf-8') as f:
    f.write(eval_text)

print(f'평가셋: {len(eval_lines)}줄, {len(eval_text.encode())} bytes')
print(f'학습셋: {len(train_lines):,}줄')

# 10% / 50% / 100% 분할 저장
split_paths = {}
for ratio, name in zip(RATIOS, RATIO_NAMES):
    n    = max(1, int(len(train_lines) * ratio))
    path = SPLIT_DIR / f'train_{name}.txt'
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(train_lines[:n]))
    split_paths[name] = path
    print(f'[{name:>4}] {n:>7,}줄 | {path.stat().st_size/1024/1024:.2f} MB')

In [ ]:
import re as _re

# 평가 함수
def evaluate_subword(tokenizer, eval_lines):
    UNK_ID = tokenizer.token_to_id('[UNK]')

    start     = time.perf_counter()
    encodings = [tokenizer.encode(line) for line in eval_lines]
    elapsed   = time.perf_counter() - start

    all_ids      = [tid for enc in encodings for tid in enc.ids]
    total_tokens = len(all_ids)
    oov_count    = all_ids.count(UNK_ID) if UNK_ID is not None else 0
    total_words  = sum(len(line.split()) for line in eval_lines)

    return {
        'vocab_size' : tokenizer.get_vocab_size(),
        'oov_rate'   : round(oov_count / total_tokens * 100, 2) if total_tokens else 0,
        'fertility'  : round(total_tokens / total_words, 3) if total_words else 0,
        'speed_ktps' : round(total_tokens / elapsed / 1000, 1) if elapsed > 0 else 0,
    }

def _is_safe_line(line):
    """Kkma NullPointerException 유발 줄 필터링"""
    line = line.strip()
    if len(line) < 10:
        return False
    # 한글 비율이 30% 미만인 줄 제외 (숫자/기호 위주 줄)
    korean = len(_re.findall(r'[가-힣]', line))
    if korean / len(line) < 0.3:
        return False
    return True

def evaluate_morpheme(tagger, eval_lines):
    safe_lines = [line for line in eval_lines if _is_safe_line(line)]

    start        = time.perf_counter()
    total_morphs = 0
    errors       = 0

    for line in safe_lines:
        try:
            total_morphs += len(tagger.morphs(line))
        except Exception:
            errors += 1

    elapsed     = time.perf_counter() - start
    total_words = sum(len(line.split()) for line in safe_lines)

    if errors:
        print(f'  [경고] {errors}줄 스킵됨')

    return {
        'fertility'  : round(total_morphs / total_words, 3) if total_words else 0,
        'speed_ktps' : round(total_morphs / elapsed / 1000, 1) if elapsed > 0 else 0,
        'elapsed_sec': round(elapsed, 2),
    }

print('평가 함수 정의 완료')

In [ ]:
bpe_results = []

for ratio, name in zip(RATIOS, RATIO_NAMES):
    tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=BPE_VOCAB_SIZE,
        special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
        min_frequency=1,
    )

    t0 = time.perf_counter()
    tokenizer.train([str(split_paths[name])], trainer)
    train_sec = round(time.perf_counter() - t0, 2)

    tokenizer.save(str(MODEL_DIR / f'bpe_{name}.json'))

    m = evaluate_subword(tokenizer, eval_lines)
    m.update({'ratio': name, 'train_sec': train_sec})
    bpe_results.append(m)
    print(f'[BPE {name}] 학습 {train_sec}s | vocab={m["vocab_size"]:,} | '
          f'OOV={m["oov_rate"]}% | fertility={m["fertility"]} | speed={m["speed_ktps"]}k tok/s')

df_bpe = pd.DataFrame(bpe_results).set_index('ratio')
print('\n=== BPE 결과 ===')
print(df_bpe[['vocab_size', 'oov_rate', 'fertility', 'speed_ktps', 'train_sec']])

In [ ]:
wp_results = []

for ratio, name in zip(RATIOS, RATIO_NAMES):
    tokenizer = Tokenizer(WordPiece(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordPieceTrainer(
        vocab_size=WP_VOCAB_SIZE,
        special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
        min_frequency=2,
    )

    t0 = time.perf_counter()
    tokenizer.train([str(split_paths[name])], trainer)
    train_sec = round(time.perf_counter() - t0, 2)

    tokenizer.save(str(MODEL_DIR / f'wordpiece_{name}.json'))

    m = evaluate_subword(tokenizer, eval_lines)
    m.update({'ratio': name, 'train_sec': train_sec})
    wp_results.append(m)
    print(f'[WP  {name}] 학습 {train_sec}s | vocab={m["vocab_size"]:,} | '
          f'OOV={m["oov_rate"]}% | fertility={m["fertility"]} | speed={m["speed_ktps"]}k tok/s')

df_wp = pd.DataFrame(wp_results).set_index('ratio')
print('\n=== WordPiece 결과 ===')
print(df_wp[['vocab_size', 'oov_rate', 'fertility', 'speed_ktps', 'train_sec']])

In [ ]:
# 형태소 분석기 비교
from konlpy.tag import Okt, Kkma
from mecab import MeCab

okt   = Okt()
kkma  = Kkma()
mecab = MeCab()

print('Okt, Kkma, Mecab 로드 완료')
print('Mecab 테스트:', mecab.morphs('대한민국 육군은 최전방에서 임무를 수행한다.'))

morph_results = {}
print('형태소 분석기 평가 중 (Kkma는 느릴 수 있음)...')

for tname, tagger in [('Mecab', mecab), ('Okt', okt), ('Kkma', kkma)]:
    r = evaluate_morpheme(tagger, eval_lines)
    morph_results[tname] = r
    print(f'[{tname:<5}] fertility={r["fertility"]} | speed={r["speed_ktps"]}k tok/s | {r["elapsed_sec"]}s')

df_morph = pd.DataFrame(morph_results).T
print('\n=== 형태소 분석기 결과 ===')
print(df_morph[['fertility', 'speed_ktps', 'elapsed_sec']])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('토크나이저 비교 실험 결과', fontsize=15, fontweight='bold')

ax = axes[0, 0]
ax.plot(RATIO_NAMES, df_bpe['vocab_size'], 'o-', label='BPE',       color='steelblue')
ax.plot(RATIO_NAMES, df_wp['vocab_size'],  's-', label='WordPiece', color='darkorange')
ax.set_title('데이터 크기별 어휘 크기'); ax.set_ylabel('어휘 크기')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(RATIO_NAMES, df_bpe['oov_rate'], 'o-', label='BPE',       color='steelblue')
ax.plot(RATIO_NAMES, df_wp['oov_rate'],  's-', label='WordPiece', color='darkorange')
ax.set_title('데이터 크기별 OOV 비율 (%)'); ax.set_ylabel('OOV (%)')
ax.legend(); ax.grid(True, alpha=0.3)

bar_names  = ['BPE\n(100%)', 'WP\n(100%)'] + list(morph_results.keys())
bar_colors = ['steelblue', 'darkorange'] + ['mediumseagreen'] * len(morph_results)
fertils    = [df_bpe.loc['100%','fertility'], df_wp.loc['100%','fertility']] + \
             [v['fertility'] for v in morph_results.values()]
ax = axes[1, 0]
bars = ax.bar(bar_names, fertils, color=bar_colors)
for b, v in zip(bars, fertils):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.2f}', ha='center', fontsize=9)
ax.set_title('Fertility 비교'); ax.set_ylabel('Fertility')
ax.grid(True, alpha=0.3, axis='y')

speeds = [df_bpe.loc['100%','speed_ktps'], df_wp.loc['100%','speed_ktps']] + \
         [v['speed_ktps'] for v in morph_results.values()]
ax = axes[1, 1]
bars = ax.bar(bar_names, speeds, color=bar_colors)
for b, v in zip(bars, speeds):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}', ha='center', fontsize=9)
ax.set_title('토크나이징 속도 (k tokens/sec)'); ax.set_ylabel('k tok/s')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('tokenizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: tokenizer_comparison.png')